This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [19]:
import great_expectations as gx
context = gx.get_context()
import logging

DEBUG:great_expectations.data_context.data_context.serializable_data_context:Searching for config file /mnt/encrypted_data/git/data-testing/great_expectations (0 layer deep)
DEBUG:great_expectations.data_context.data_context.serializable_data_context:Searching for config file /mnt/encrypted_data/git/data-testing (1 layer deep)
DEBUG:great_expectations.data_context.data_context.serializable_data_context:Found config file at /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
DEBUG:great_expectations.data_context.data_context.serializable_data_context:Using project config: /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml
INFO:great_expectations.data_context.data_context.file_data_context:FileDataContext loading fluent config
DEBUG:great_expectations.datasource.fluent.config:loaded from yaml ->
{'anonymous_usage_statistics': {'data_context_id': '0b565257-5608-48e9-8b6d-8bbf5d4945d9',
                                'enabled': False}

In [2]:
import yaml

In [3]:
from datetime import date

In [4]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [5]:
connection_string = """bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json"""

In [6]:
with open("datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [7]:
datasource_config.get("project")

'gfw-google-827'

In [8]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.data

In [9]:
gx_datasource.get_asset_names()

{'messages-2.5',
 'messages-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}

In [10]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.messages_positions.2-5',
 'gfw-google-827.alerts.messages_positions.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_info.2-5',
 'gfw-google-827.alerts.segment_info.3-0-0',
 'gfw-google-827.alerts.segment_vessel.2-5',
 'gfw-google-827.alerts.segment_vessel.3-0-0',
 'gfw-google-827.alerts.segs_activity.2-5',
 'gfw-google-827.alerts.segs_activity.3-0-0',
 'gfw-google-827.alerts.segs_activity_daily.2-5',
 'gfw-google-827.alerts.segs_activity_daily.3-0-0',
 'gfw-google-827.alerts.ssvids_identities.2-5',
 'gfw-google-827.alerts.ssvids_identities.3-0-0',
 'gfw-google-827.alerts.ssvids_identities_daily.2-5',
 'gfw-google-827.alerts.ssvids_identities_daily.3-0-0',
 'gfw-google-827.alerts.stats_daily.2-5',
 'gfw-google-827.alerts.stats_daily.3-0-0',
 'gfw-google-827.alerts.vessel_info.2-5',


In [18]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if 'segs_activity_daily' in es and 'constraints' in es]:
    current_expectation_suite = context.get_expectation_suite(current_expectation_suite_name)
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        # if gx_asset.batch_metadata.get('splitter_date_type') == 'TIMESTAMP':
        #     DATE_PARTITION_VALUE=datetime.fromisoformat('2020-01-01T00:00:00+00:00')
        # else:
        #     DATE_PARTITION_VALUE=date.fromisoformat('2020-01-01')
        br_options={DATE_PARTITION_COLUMN: '2023-04-01'}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    context.add_checkpoint(
        name=f"{current_expectation_suite_name}-checkpoint",
        validations=[
        {
        "batch_request": gx_br
        }
        ],
        expectation_suite_name=current_expectation_suite_name
    )

DEBUG:great_expectations.data_context.util:(instantiate_class_from_config) module_name -> great_expectations.checkpoint
DEBUG:great_expectations.data_context.util:(instantiate_class_from_config) module_name -> great_expectations.checkpoint


In [23]:
context.list_checkpoints()

gfw-google-827.constraints.segs_activity_daily.2-5-checkpoint
gfw-google-827.constraints.segs_activity_daily.3-0-0-checkpoint
